# 第 2 章：推理性能基础

这个 Notebook 先用简单算术估算 Decode 的带宽上限；如果 Colab 运行时已启用 GPU，再做一个小型矩阵乘基准。

In [ ]:
# 估算 8B FP16 在 H100 上的 batch=1 Decode 上限
parameters = 8e9
bytes_per_parameter = 2  # FP16
h100_bandwidth = 3.35e12 # bytes/s
weight_bytes = parameters * bytes_per_parameter
upper_bound = h100_bandwidth / weight_bytes
print(f'权重大小约 {weight_bytes / 1e9:.0f} GB')
print(f'带宽理论上限约 {upper_bound:.0f} token/s')
print('这是理想上限：真实系统还会有 KV、kernel 启动和采样开销。')

## 可选：比较一次 token 与一段 token

在 Colab 菜单中选择 **运行时 → 更改运行时类型 → T4 GPU** 后再运行。`S=1` 类似 Decode，`S=1024` 类似 Prefill。这里用小矩阵避免免费 GPU 内存不足。

In [ ]:
import time
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print('设备:', device)

D, D_FF = 512, 1536
ffn = nn.Sequential(nn.Linear(D, D_FF, bias=False), nn.SiLU(), nn.Linear(D_FF, D, bias=False)).to(device, dtype)

def bench(seq_len, rounds=30):
    x = torch.randn(1, seq_len, D, device=device, dtype=dtype)
    for _ in range(5):
        ffn(x)
    if device == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(rounds):
        ffn(x)
    if device == 'cuda': torch.cuda.synchronize()
    print(f'S={seq_len:4d}: {(time.perf_counter() - start) / rounds * 1000:.2f} ms')

bench(1)
bench(1024)